In [60]:
import numpy as np
import pandas as pd

df = pd.read_csv('../dataset/raw/Customer-Churn-Records.csv')
print(df.head(5))

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  Complain  Satisfaction Score Card Type  \
0        101348.88       1         1                   2   DIAMOND   
1        112542.58

In [61]:
print("All columns:")
print(df.columns.tolist())
print(f"\nTotal columns: {len(df.columns)}")

All columns:
['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Complain', 'Satisfaction Score', 'Card Type', 'Point Earned']

Total columns: 18


In [62]:
# Removing columns that are not useful for modeling
columns_to_drop = ['RowNumber', 'CustomerId', 'Surname', 'Card Type']
df_cleaned = df.drop(columns=columns_to_drop)
print("\nColumns after dropping:")
print(df_cleaned.columns.tolist())



Columns after dropping:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Complain', 'Satisfaction Score', 'Point Earned']


In [63]:
print("Data types:")
print(df_cleaned.dtypes)

Data types:
CreditScore             int64
Geography              object
Gender                 object
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Complain                int64
Satisfaction Score      int64
Point Earned            int64
dtype: object


In [64]:
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df_cleaned.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Removing target from numerical list
numerical_cols.remove('Exited')

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

Categorical columns: ['Geography', 'Gender']
Numerical columns: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Complain', 'Satisfaction Score', 'Point Earned']


In [65]:
for col in categorical_cols:
    print(f"{df_cleaned[col].value_counts()}\n")

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

Gender
Male      5457
Female    4543
Name: count, dtype: int64



In [66]:
# Encoding categorical variables Geography and Gender
# One-hot encode Geography 
geography_dummies = pd.get_dummies(df_cleaned['Geography'], prefix='Geography')
print("Geography encoding:")
print(geography_dummies.head())

Geography encoding:
   Geography_France  Geography_Germany  Geography_Spain
0              True              False            False
1             False              False             True
2              True              False            False
3              True              False            False
4             False              False             True


In [67]:
df_cleaned['Gender'] = df_cleaned['Gender'].map({'Female': 0, 'Male': 1})
print(f"\nGender after encoding:")
print(df_cleaned['Gender'].value_counts())


Gender after encoding:
Gender
1    5457
0    4543
Name: count, dtype: int64


In [68]:
# Combine and drop original Geography
df_clean = pd.concat([df_cleaned, geography_dummies], axis=1)
df_clean = df_clean.drop(columns=['Geography'])

In [69]:
print(f"\nFinal shape: {df_clean.shape}")
print(f"\nFinal columns: {df_clean.columns.tolist()}")


Final shape: (10000, 16)

Final columns: ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Complain', 'Satisfaction Score', 'Point Earned', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


In [70]:
print("Final dataset check:")
print(df_clean.dtypes)
print(f"\nMissing values: {df_clean.isnull().sum().sum()}")
print(f"\nFirst 5 rows:")
df_clean.head()

Final dataset check:
CreditScore             int64
Gender                  int64
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Complain                int64
Satisfaction Score      int64
Point Earned            int64
Geography_France         bool
Geography_Germany        bool
Geography_Spain          bool
dtype: object

Missing values: 0

First 5 rows:


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Point Earned,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,2,464,True,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,0,1,3,456,False,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,3,377,True,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,0,0,5,350,True,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,5,425,False,False,True


In [74]:
# Save the cleaned dataset
df_clean.to_csv('../dataset/processed/Customer-churn-clean.csv', index=False)